In [1]:
# 在notebooks目录下创建这个文件
# 内容如下：

"""
# Quant-MVP 数据模块演示 Notebook

这个Notebook演示如何使用数据模块获取和处理金融数据。
"""

# 1. 设置环境
import sys
import os

# 添加项目根目录到Python路径
project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, project_root)

# 2. 导入模块
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from src.data import data_manager, data_cache

# 设置中文字体和图表样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ 环境设置完成")

# 3. 获取单个股票历史数据
symbol = 'AAPL'
start_date = '2023-01-01'
end_date = '2023-12-31'

print(f"获取 {symbol} 的历史数据...")
df = data_manager.get_historical_data(symbol, start_date, end_date)

print(f"数据形状: {df.shape}")
print(f"时间范围: {df.index[0]} 到 {df.index[-1]}")

# 显示前几行数据
df.head()

# 4. 数据可视化
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 价格走势
axes[0, 0].plot(df.index, df['close'], label='收盘价', linewidth=2)
axes[0, 0].set_title(f'{symbol} 收盘价走势')
axes[0, 0].set_xlabel('日期')
axes[0, 0].set_ylabel('价格 ($)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 成交量
axes[0, 1].bar(df.index, df['volume'], alpha=0.6, color='orange')
axes[0, 1].set_title(f'{symbol} 成交量')
axes[0, 1].set_xlabel('日期')
axes[0, 1].set_ylabel('成交量')
axes[0, 1].grid(True, alpha=0.3)

# 价格分布
axes[1, 0].hist(df['close'], bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_title(f'{symbol} 收盘价分布')
axes[1, 0].set_xlabel('价格 ($)')
axes[1, 0].set_ylabel('频次')

# 日收益率
daily_returns = df['close'].pct_change().dropna()
axes[1, 1].hist(daily_returns, bins=50, edgecolor='black', alpha=0.7)
axes[1, 1].set_title(f'{symbol} 日收益率分布')
axes[1, 1].set_xlabel('日收益率')
axes[1, 1].set_ylabel('频次')

plt.tight_layout()
plt.show()

# 5. 获取多个股票数据
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
start_date = '2023-01-01'
end_date = '2023-06-30'

print(f"批量获取 {len(symbols)} 个股票的数据...")
all_data = data_manager.get_multiple_symbols_data(symbols, start_date, end_date)

# 6. 分析多个股票
returns_data = {}
for symbol, df in all_data.items():
    if not df.empty and 'close' in df.columns:
        # 计算累计收益率
        prices = df['close']
        returns = prices.pct_change().dropna()
        cumulative_returns = (1 + returns).cumprod() - 1
        
        returns_data[symbol] = {
            'prices': prices,
            'returns': returns,
            'cumulative_returns': cumulative_returns
        }

# 绘制多个股票的累计收益率
plt.figure(figsize=(12, 6))
for symbol, data in returns_data.items():
    if 'cumulative_returns' in data and not data['cumulative_returns'].empty:
        plt.plot(data['cumulative_returns'].index, 
                data['cumulative_returns'].values * 100, 
                label=symbol, linewidth=2)

plt.title('多个股票的累计收益率对比 (2023上半年)')
plt.xlabel('日期')
plt.ylabel('累计收益率 (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 7. 缓存统计
print("缓存统计信息:")
stats = data_cache.get_stats()
for key, value in stats.items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for subkey, subvalue in value.items():
            print(f"    {subkey}: {subvalue}")
    else:
        print(f"  {key}: {value}")

# 8. 获取实时数据（如果交易时间）
print("\n尝试获取实时数据...")
real_time_data = {}
for symbol in ['AAPL', 'MSFT']:
    data = data_manager.get_realtime_data(symbol)
    if data:
        real_time_data[symbol] = data

if real_time_data:
    print("实时数据:")
    for symbol, data in real_time_data.items():
        print(f"  {symbol}: ${data.get('price', 'N/A')} "
              f"({data.get('change_percent', 'N/A')}%)")
else:
    print("⚠ 非交易时间，无实时数据")

# 9. 关闭数据管理器（释放资源）
data_manager.close()
print("\n✅ 演示完成！")

NameError: name '__file__' is not defined